# A2 — Phase 5 layout, Phase 6 OCR, and Phase 7 chunking

This notebook is a Kaggle evidence runner for branch `2105001/phases-5-7`. It does not create Git commits or push to GitHub. Save a Kaggle version with outputs, download this `.ipynb`, and commit the executed file from the local repository. The existing Qwen3-VL drafts are evaluated from `grading_kit/labels.jsonl`; the expensive Qwen model is not rerun.

In [ ]:
from pathlib import Path
import os, subprocess, sys

REPO_URL = "https://github.com/FAHIM-ISHTIAK/doc-agent-20.git"
REPO_BRANCH = "2105001/phases-5-7"
REPO_DIR = Path("/kaggle/working/doc-agent-20-phase57")
ARTIFACT_DIR = Path("/kaggle/working/artifacts/phase57")

def run_command(command, cwd=None):
    print("$", " ".join(map(str, command)))
    result = subprocess.run([str(part) for part in command], cwd=str(cwd) if cwd else None, text=True, capture_output=True, check=False)
    if result.stdout.strip(): print(result.stdout.strip())
    if result.stderr.strip(): print(result.stderr.strip())
    if result.returncode != 0: raise RuntimeError(f"Command failed ({result.returncode}): {command}")
    return result

if REPO_DIR.exists():
    if not (REPO_DIR / ".git").is_dir(): raise RuntimeError(f"Refusing to replace non-Git path: {REPO_DIR}")
    if run_command(["git", "status", "--porcelain"], REPO_DIR).stdout.strip(): raise RuntimeError("Existing Kaggle clone is dirty; start a fresh session or use a new REPO_DIR")
    run_command(["git", "fetch", "origin"], REPO_DIR)
    run_command(["git", "switch", REPO_BRANCH], REPO_DIR)
    run_command(["git", "pull", "--ff-only", "origin", REPO_BRANCH], REPO_DIR)
else:
    run_command(["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)])
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(REPO_DIR)
commit_hash = run_command(["git", "rev-parse", "HEAD"], REPO_DIR).stdout.strip()
print("TESTED GIT COMMIT:", commit_hash)
print("Kaggle creates no Git commit in this workflow.")

In [ ]:
PHASE57_PACKAGES = ["pydantic==2.12.3", "Pillow==11.3.0", "PyYAML==6.0.3", "PyMuPDF==1.26.7", "pytest==8.4.2", "easyocr==1.7.2"]
run_command([sys.executable, "-m", "pip", "install", *PHASE57_PACKAGES])
run_command([sys.executable, "-m", "pip", "install", "-e", str(REPO_DIR), "--no-deps"])
if str(REPO_DIR / "src") not in sys.path: sys.path.insert(0, str(REPO_DIR / "src"))
test_result = run_command([sys.executable, "-m", "pytest", "tests/test_ingest.py", "tests/test_ocr.py", "tests/test_retrieval.py", "tests/test_contracts.py", "-q"], REPO_DIR)
print("PHASE 5-7 FOCUSED TESTS: PASS")

## Phase 5 — Real-page layout and reading order

In [ ]:
import json
from copy import deepcopy
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image
from doc_agent import config
from doc_agent.contracts import Page
from doc_agent.vision import layout

cfg = config.load(REPO_DIR / "configs/config.yaml")
cfg = deepcopy(cfg)
cfg["device"] = "cuda" if __import__("torch").cuda.is_available() else "cpu"
cfg["data"]["artifact_dir"] = str(ARTIFACT_DIR)
labels = [json.loads(line) for line in (REPO_DIR / "grading_kit/labels.jsonl").read_text(encoding="utf-8").splitlines() if line.strip()]
pages_dir = REPO_DIR / "grading_kit/heldout_pages"
pages = [Page(id=item["page_id"], doc_id=item["doc_id"], image_path=str((pages_dir / item["image_filename"]).resolve())) for item in labels]
regions = layout.detect(pages, cfg)
regions_by_page = {page.id: [region for region in regions if region.page_id == page.id] for page in pages}
assert all(regions_by_page[page.id] for page in pages)
for page in pages:
    with Image.open(page.image_path) as image:
        width, height = image.size
    assert all(0 <= r.bbox[0] < r.bbox[2] <= width and 0 <= r.bbox[1] < r.bbox[3] <= height for r in regions_by_page[page.id])
print("Pages:", len(pages), "Regions:", len(regions))
print("Kinds:", {kind: sum(r.kind == kind for r in regions) for kind in ["heading", "text", "table", "figure"]})
print("PHASE 5 REAL-PAGE LAYOUT CONTRACT: PASS")

In [ ]:
sample_ids = ["bhasha_prakash_1942_p0050", "bhasha_prakash_1942_p0170", "nctb_bangla_grammar_2026_p0020", "nctb_bangla_grammar_2026_p0035", "nctb_bangla_grammar_2026_p0075"]
sample_pages = [page for page in pages if page.id in sample_ids]
figure, axes = plt.subplots(len(sample_pages), 1, figsize=(14, 8 * len(sample_pages)))
if len(sample_pages) == 1: axes = [axes]
colors = {"heading": "blue", "text": "red", "table": "green", "figure": "purple"}
for axis, page in zip(axes, sample_pages):
    with Image.open(page.image_path) as source: preview = source.copy()
    axis.imshow(preview, cmap="gray")
    for order, region in enumerate(regions_by_page[page.id], start=1):
        x0, y0, x1, y1 = region.bbox
        axis.add_patch(Rectangle((x0, y0), x1-x0, y1-y0, fill=False, linewidth=2, edgecolor=colors[region.kind]))
        axis.text(x0, y0, str(order), color="white", fontsize=9, bbox={"facecolor": colors[region.kind], "alpha": 0.8})
    axis.set_title(f"{page.id}: numbered reading order")
    axis.axis("off")
plt.tight_layout(); plt.show()
print("Visually verify region coverage, rule/example order, and atomic tables.")
print("PHASE 5 LAYOUT GALLERY: READY FOR HUMAN CHECK")

## Phase 6 — EasyOCR pipeline and stored-candidate comparison

In [ ]:
import time, unicodedata
from doc_agent.vision import ocr

def normalize_for_metric(text):
    return " ".join(unicodedata.normalize("NFC", text).split())

def edit_distance(left, right):
    previous = list(range(len(right) + 1))
    for i, a in enumerate(left, start=1):
        current = [i]
        for j, b in enumerate(right, start=1): current.append(min(current[-1]+1, previous[j]+1, previous[j-1]+(a != b)))
        previous = current
    return previous[-1]

def error_rates(predictions, golds, normalized=False):
    pairs = [(normalize_for_metric(p), normalize_for_metric(g)) if normalized else (p, g) for p, g in zip(predictions, golds)]
    char_errors = sum(edit_distance(list(p), list(g)) for p, g in pairs)
    char_total = sum(len(g) for _, g in pairs)
    word_errors = sum(edit_distance(p.split(), g.split()) for p, g in pairs)
    word_total = sum(len(g.split()) for _, g in pairs)
    return {"cer": char_errors/max(char_total, 1), "wer": word_errors/max(word_total, 1), "characters": char_total, "words": word_total}

reader = ocr.Reader(cfg)
pipeline_predictions = {}
started = time.perf_counter()
for page in pages:
    raw_parts = [reader.transcribe_region(region).strip() for region in regions_by_page[page.id]]
    pipeline_predictions[page.id] = "\n".join(part for part in raw_parts if part)
elapsed = time.perf_counter() - started
assert all(pipeline_predictions[page.id].strip() for page in pages)
print(f"EasyOCR layout pipeline processed {len(pages)} pages in {elapsed:.2f}s on {cfg['device']}")

In [ ]:
gold = {item["page_id"]: item["text"] for item in labels}
stored_candidate = {item["page_id"]: (item.get("qwen3_vl_8b_draft") or item.get("easyocr_draft") or item["ocr_draft"]) for item in labels}
systems = {"easyocr_layout_pipeline": pipeline_predictions, "stored_phase3_candidate": stored_candidate}
metric_rows = []
for system_name, predictions in systems.items():
    for doc_id in sorted({item["doc_id"] for item in labels}) + ["overall"]:
        selected = [item for item in labels if doc_id == "overall" or item["doc_id"] == doc_id]
        predicted = [predictions[item["page_id"]] for item in selected]
        references = [gold[item["page_id"]] for item in selected]
        for normalized in (False, True):
            row = {"system": system_name, "document": doc_id, "normalization": "NFC+whitespace" if normalized else "raw", "pages": len(selected), **error_rates(predicted, references, normalized)}
            metric_rows.append(row)
for row in metric_rows: print(row)
worst = max(labels, key=lambda item: error_rates([pipeline_predictions[item["page_id"]]], [item["text"]], True)["cer"])
print("WORST EASY-OCR PAGE:", worst["page_id"], worst["categories"])
print("Prediction preview:", pipeline_predictions[worst["page_id"]][:600])
print("PHASE 6 RAW/NORMALIZED OCR METRICS: PASS")

## Phase 6 — Full-corpus usable OCR word count

Attach the private dataset containing `bhasha_prakash_1942.pdf` and `nctb_bangla_grammar.pdf`. This section renders and OCRs all declared pages and may take a substantial part of a GPU session.

In [ ]:
import gc
from doc_agent.ingest import loader, preprocess

private_input = Path("/kaggle/input")
bhasha_matches = list(private_input.rglob("bhasha_prakash_1942.pdf"))
nctb_matches = list(private_input.rglob("nctb_bangla_grammar.pdf"))
if len(bhasha_matches) != 1 or len(nctb_matches) != 1:
    raise RuntimeError(f"Attach exactly one private copy of each corpus PDF; found {bhasha_matches=} {nctb_matches=}")
full_raw_dir = Path("/kaggle/working/data/raw")
full_interim_dir = Path("/kaggle/working/data/interim")
render_env = os.environ.copy()
render_env.update({"BMA_BHASHA_PDF": str(bhasha_matches[0]), "BMA_NCTB_PDF": str(nctb_matches[0]), "BMA_RAW_DIR": str(full_raw_dir), "BMA_INTERIM_DIR": str(full_interim_dir), "BMA_MAX_PAGES": "0", "BMA_RENDER_DPI": "300"})
print("$ bash scripts/get_data.sh")
render_result = subprocess.run(["bash", "scripts/get_data.sh"], cwd=REPO_DIR, env=render_env, text=True, capture_output=True, check=False)
print(render_result.stdout); print(render_result.stderr)
if render_result.returncode != 0: raise RuntimeError("Full corpus rendering failed")
full_cfg = deepcopy(cfg)
full_cfg["runtime"]["mode"] = "full"
full_cfg["data"]["raw_dir"] = str(full_raw_dir)
full_cfg["data"]["artifact_dir"] = str(ARTIFACT_DIR / "full")
saved_env = {name: os.environ.get(name) for name in ["DOC_AGENT_DATA_DIR", "DOC_AGENT_ARTIFACT_DIR", "DOC_AGENT_RUN_MODE"]}
os.environ["DOC_AGENT_DATA_DIR"] = str(full_raw_dir)
os.environ["DOC_AGENT_ARTIFACT_DIR"] = str(ARTIFACT_DIR / "full")
os.environ["DOC_AGENT_RUN_MODE"] = "full"
del reader
gc.collect()
if __import__("torch").cuda.is_available(): __import__("torch").cuda.empty_cache()
try:
    full_pages = loader.load_pages(full_cfg)
    full_clean_pages = preprocess.run(full_pages, full_cfg)
    full_regions = layout.detect(full_clean_pages, full_cfg)
    full_chunks = ocr.transcribe(full_regions, full_cfg)
finally:
    for name, value in saved_env.items():
        if value is None: os.environ.pop(name, None)
        else: os.environ[name] = value
full_word_count = sum(len(item.text.split()) for item in full_chunks)
covered_pages = {page_id for item in full_chunks for page_id in item.page_ids}
full_metrics = {"raw_declared_pages": 783, "usable_loaded_pages": len(full_pages), "ocr_covered_pages": len(covered_pages), "ocr_regions": len(full_chunks), "usable_ocr_words": full_word_count}
full_metrics_path = ARTIFACT_DIR / "full_ocr_metrics.json"
full_metrics_path.write_text(json.dumps(full_metrics, ensure_ascii=False, indent=2), encoding="utf-8")
full_ocr_path = ARTIFACT_DIR / "full_ocr_chunks.jsonl"
full_ocr_path.write_text("".join(json.dumps(item.model_dump(), ensure_ascii=False) + "\n" for item in full_chunks), encoding="utf-8")
print(json.dumps(full_metrics, ensure_ascii=False, indent=2))
assert len(full_pages) >= 300 and full_word_count >= 60000
print("PHASE 6 FULL-CORPUS WORD COUNT: PASS")
print("Temporary evidence files:", full_metrics_path, full_ocr_path)

## Phase 7 — Fixed versus grammar-rule-aware chunking

In [ ]:
from doc_agent.contracts import Chunk
from doc_agent.index import chunk as chunk_stage

source_chunks = full_chunks
fixed_cfg, rule_cfg = deepcopy(cfg), deepcopy(cfg)
fixed_cfg["index"]["rule_aware"] = False
rule_cfg["index"]["rule_aware"] = True
fixed_chunks = chunk_stage.split(source_chunks, fixed_cfg)
rule_chunks = chunk_stage.split(source_chunks, rule_cfg)
print("Fixed chunks:", len(fixed_chunks), "Rule-aware chunks:", len(rule_chunks))
assert all(item.page_ids for item in fixed_chunks + rule_chunks)
assert set(page_id for item in rule_chunks for page_id in item.page_ids) == covered_pages
for heading, values in [("FIXED", fixed_chunks[:3]), ("RULE-AWARE", rule_chunks[:3])]:
    print("\n", heading)
    for item in values: print({"id": item.id, "page_ids": item.page_ids, "text": item.text[:500]})
print("PHASE 7 CHUNKING COMPARISON: PASS")
print("Embedding, FAISS persistence, and retrieval evidence are intentionally outside this chunking work package.")